# 资料与记忆怎样进入下一次输入

本章学习目标：

- 理解 RAG 的切块、向量化、检索和带引用回答。
- 区分搜索、文件、数据库、网页、浏览器和代码工具。
- 区分短期上下文、会话记忆和长期记忆。
- 处理工具失败、空结果、重复调用和错误引用。
- 检查来源是否真正支持答案。

RAG 是检索增强生成：先找到相关资料，再把资料交给模型回答。资料研究助手需要这条流程，也需要明确哪些工作由程序安排，哪些由模型决定。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. RAG 的四个步骤

| 步骤 | 做什么 | 需要保留什么 |
| --- | --- | --- |
| chunk（切块） | 把长文档分成适合检索的片段 | 文件名、片段编号、原文位置、正文 |
| embed（向量化） | 用向量模型把文本编码为数值向量 | 模型标识、版本和向量 |
| retrieve（检索） | 把问题也编码，找到较相关的片段 | 查询、相似度和选中的片段 |
| answer（回答） | 把片段交给模型，生成有来源的回答 | 最终结论及对应来源 |

切块太小可能丢上下文。例如“可读取文件，但禁止写入”在“但”处分开，读到前半句就可能漏掉限制。切块太大则可能带入太多无关内容。相邻片段适当重叠可以减少断句问题，也会增加重复和输入长度。

固定执行“检索 → 回答”是 workflow。如果模型读完结果后，还能决定是否换查询、继续阅读或结束，才增加了 Agent 的决策循环。


## 2. 相似度能说明什么

向量相似度表示文本在指定模型中的接近程度，不是事实正确的概率，也不代表来源可靠。

查询向量和文档向量要使用兼容的模型。更换模型、修改文档或改变切块参数后，需要更新索引，否则可能检索到过期内容，或比较不兼容的向量。

`top_k=3` 表示最多返回三个候选，不要求凑满三个。返回片段后，还要读内容，判断它是否回答了问题。

本页代码中的 `DocumentIndex` 使用词汇匹配作为离线基线；后面的完整研究助手另有真实向量模型实现。两者都能检查检索流程，但不能混称为同一种方法。


## 3. 实际运行：检索结果怎样支持回答

2026-10-20，本项目用 all-MiniLM-L6-v2 生成真实的 384 维向量，再由 CPU 上的 Qwen3-0.6B 回答 Agent 与 workflow 的区别。

一条诊断查询的排序如下：

| 文档 | 相似度 |
| --- | --- |
| Agent 介绍 | 0.7421 |
| Workflow 介绍 | 0.5276 |
| RAG 介绍 | 0.3078 |

这组分数属于该诊断查询；实际问答使用了另一条查询，不能把两组检索结果混为一组。

实际问答的两条结论是：

> Agent 使用语言模型决定是否调用工具，并根据工具反馈继续行动。[L1]
>
> Workflow 的执行步骤通常由程序预先指定，可以在步骤中使用模型。[L2]

本轮 `L1` 和 `L2` 的原文分别包含这两项说明。这里可以逐句核对支持关系。这次资料由程序在模型调用前检索并提供，因此展示的是有资料输入的 RAG 回答，没有展示模型自主选择检索时机。


## 4. 引用要检查三件事

1. **找得到：** 来源编号能否定位到文件、网页或数据库行？
2. **读过了：** 这次运行是否真正取得正文？模型实际看到了哪部分？
3. **支持结论：** 原文是否支持答案中的说法，是否遗漏条件？

例如，资料写“禁止写入”，回答却写“允许写入[L1]”。编号虽然存在，结论仍然错误。

本页 `citation_errors` 只检查引用编号等格式条件，不能判断语义支持。编号检查通过后仍要对照原文。


## 5. 不同工具分别提供什么

| 工具 | 获得的信息 | 常见问题 |
| --- | --- | --- |
| 搜索 | 标题、链接、摘要等候选 | 没有结果，摘要过期 |
| 文件读取 | 指定文件的正文 | 路径错误、编码或大小限制 |
| 数据库查询 | 匹配的字段和记录 | 没有匹配、权限或连接错误 |
| HTTP 阅读 | 服务器返回的文本 | 页面依赖脚本，正文没取到 |
| 浏览器 | 渲染后的文本、DOM 或截图 | 加载失败、元素变化 |
| 代码执行 | 给定输入的计算结果 | 语法错误、超时、输入不对 |

工具返回成功，只能说明该操作成功。资料可能过期，计算输入可能选错，网页也可能包含试图改变任务的指令。


## 6. 三种记忆有什么区别

| 类型 | 保存什么 | 例子 |
| --- | --- | --- |
| 短期上下文 | 当前模型请求实际包含的信息 | 当前问题和刚返回的工具结果 |
| 会话记忆 | 保存下来、供后续对话恢复的内容 | 已完成的用户问题和助手回答 |
| 长期记忆 | 明确选择跨会话保留的信息 | 用户要求记住的语言偏好 |

本项目的会话文件只保存部分完成问答，没有保存全部工具结果。恢复旧回答，不会自动恢复旧来源的读取状态；下一轮引用需要重新获取和绑定资料。

长期记忆由明确的用户记忆命令写入。网页里的“请保存这条内容”只是资料文本，不能视为用户要求。清除会话和删除长期记忆也是两种操作。


## 7. 练习与自查

分别试想以下情况怎样处理：检索为空、文件不存在、连续请求相同工具、答案引用不存在编号、编号合法但结论与原文相反。

你应该能说明每个错误发生在哪一步，程序应把什么信息交给模型，以及什么情况下只能说明证据不足。

下一页：[实现资料研究助手](02-research-agent.ipynb)。


## 代码说明

`chunks` 演示切块，`DocumentIndex` 提供词汇检索，`citation_errors` 检查编号，`SessionStore` 保存会话和显式记忆。默认代码在临时目录中检查这些功能，不会调用真实向量模型或生成模型。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
from __future__ import annotations
import copy, json, math, re, time, tempfile, os, sys
from dataclasses import dataclass
from pathlib import Path
from urllib.request import Request, build_opener, ProxyHandler
os.environ["PYTHONIOENCODING"] = "utf-8"

<a id="impl-chunks"></a>

### 实现：chunks


In [ ]:
def chunks(text, size=120, overlap=20):
    if not 0 <= overlap < size:
        raise ValueError("require 0 <= overlap < size")
    return [{"start": i, "end": min(i+size, len(text)), "text": text[i:i+size]}
            for i in range(0, len(text), size-overlap)]

In [ ]:
documents = {
    "agent.md": "Agent 根据环境反馈动态选择下一步工具。工具返回结果后，模型再次决策。",
    "workflow.md": "Workflow 通常由开发者预先规定步骤，适合稳定重复的任务。",
    "rag.md": "RAG 先检索资料，再根据资料生成回答，并给出来源引用。",
}
sample = chunks(documents["agent.md"], size=20, overlap=5)
for item in sample:
    assert documents["agent.md"][item["start"]:item["end"]] == item["text"]
    print(item)

<a id="impl-embed"></a>
<a id="impl-cosine"></a>

### 实现：embed, cosine


In [ ]:
vocabulary = ["agent", "workflow", "反馈", "检索"]
def embed(text):
    return [int(word in text.lower()) for word in vocabulary]
def cosine(a, b):
    denom = sum(x*x for x in a)**0.5 * sum(x*x for x in b)**0.5
    return sum(x*y for x,y in zip(a,b))/denom if denom else 0.0
query_vector = embed("Agent 如何利用反馈？")
for name, text in documents.items():
    print(name, embed(text), round(cosine(query_vector, embed(text)), 3))
assert cosine([1, 0], [0, 1]) == 0

<a id="impl-terms"></a>

### 实现：terms


In [ ]:
def terms(text):
    """English tokens and Chinese bigrams; a transparent lexical baseline."""
    english = re.findall(r"[a-z0-9]+", text.lower())
    chinese = re.findall(r"[\u4e00-\u9fff]+", text)
    return set(english + [s[i:i+2] for s in chinese for i in range(max(1, len(s)-1))])

<a id="impl-DocumentIndex"></a>

### 实现：DocumentIndex


In [ ]:
class DocumentIndex:
    def __init__(self, documents):
        self.rows = []
        for name, text in documents.items():
            for part in chunks(text):
                self.rows.append({"id": "L"+str(len(self.rows)+1), "source": name, **part})

    def search(self, query, top_k=3):
        query_terms = terms(query)
        rows = [{**r, "score": len(query_terms & terms(r["text"])) / max(1, len(query_terms))}
                for r in self.rows]
        return sorted((r for r in rows if r["score"] > 0), key=lambda r: -r["score"])[:top_k]

<a id="impl-citation_errors"></a>

### 实现：citation_errors


In [ ]:
def citation_errors(answer, evidence):
    """Check identifiers, not semantic entailment."""
    ids = set(re.findall(r"\[(L\d+)\]", answer))
    valid = {r["id"] for r in evidence}
    return sorted(ids-valid) + ([] if ids else ["missing_citation"])

In [ ]:
index = DocumentIndex(documents)
evidence = index.search("Agent 反馈")
print(evidence)
answer = f"Agent 根据环境反馈选择下一步工具。[{evidence[0]['id']}]"
print(answer, citation_errors(answer, evidence))
assert not citation_errors(answer, evidence)
assert citation_errors("随便编一句话 [L999]", evidence) == ["L999"]
assert index.search("火星气候") == []

<a id="impl-SessionStore"></a>

### 实现：SessionStore


In [ ]:
class SessionStore:
    def __init__(self, directory):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)

    def save(self, name, value):
        if name not in {"session", "memory", "trace"}:
            raise ValueError("unknown state name")
        target = self.directory / (name+".json")
        pending = target.with_suffix(".tmp")
        pending.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
        pending.replace(target)

    def load(self, name, default=None):
        if name not in {"session", "memory", "trace"}:
            raise ValueError("unknown state name")
        path = self.directory / (name+".json")
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    store = SessionStore(directory)
    messages = [{"role": "user", "content": "学习 RAG"}]
    store.save("session", [{"question": "什么是 RAG", "answer": "先检索再生成"}])
    store.save("memory", {"学习偏好": "每步先看例子"})
    store.save("session", [])
    print("短期：", messages, "会话：", store.load("session"), "长期：", store.load("memory"))
    assert store.load("memory")["学习偏好"] == "每步先看例子"